# CAMELYON16 feature preprocessing on Kaggle
Attach the preprocessing bundle and one raw feature dataset. For **multiscale**, also attach the small supplemental dataset containing the 17 missing `tumor_095`–`tumor_111` CSV files. The notebook combines both inputs with file links in `/kaggle/temp`; it does not copy the raw features or change either Kaggle input. Set `SOURCE` below and run all cells.

After an interactive test succeeds, choose **Save Version → Save & Run All**, wait for the saved run to finish, and inspect its Output tab. Quick Save does not persist the prepared files.


In [ ]:
SOURCE = "multiscale"  # uniscale, multiscale, or uni
SEED = 42


In [ ]:
%%writefile /kaggle/working/multiscale_overlay.py
"""Build a no-copy view of a partial multiscale Kaggle input plus a supplement."""

from __future__ import annotations

import csv
from pathlib import Path


def overlay_multiscale(primary: Path, input_base: Path, reference: Path, overlay: Path) -> tuple[Path, list[str]]:
    with reference.open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    expected = {r["slide_id"]: int(r["label"]) for r in rows}
    if len(rows) != 399 or len(expected) != 399:
        raise ValueError("Expected 399 unique reference slide IDs")

    original = {}
    for folder in ("0-normal", "1-tumor"):
        for path in (primary / folder).glob("*.csv"):
            if path.stem in original:
                raise ValueError(f"Duplicate primary slide: {path.stem}")
            original[path.stem] = path
    if set(original) - set(expected):
        raise ValueError(f"Unexpected primary slides: {sorted(set(original)-set(expected))}")
    missing = sorted(set(expected) - set(original))
    if not missing:
        return primary, []

    supplemental = {}
    for path in input_base.rglob("*.csv"):
        if path.stem not in missing:
            continue
        if path.is_relative_to(primary):
            continue
        if path.stem in supplemental:
            raise ValueError(f"Duplicate supplemental slide: {path.stem}")
        supplemental[path.stem] = path
    if set(supplemental) != set(missing):
        raise ValueError(f"Supplement incomplete; missing slide IDs: {sorted(set(missing)-set(supplemental))}")

    for sid, path in supplemental.items():
        expected_folder = "1-tumor" if expected[sid] else "0-normal"
        if (path.parent.name in ("0-normal", "1-tumor") and
                path.parent.name != expected_folder) or path.stat().st_size == 0:
            raise ValueError(f"Wrong class folder or empty supplement: {path}")
        with path.open(newline="") as handle:
            if len(next(csv.reader(handle))) != 1024:
                raise ValueError(f"Wrong feature width in supplement: {path}")

    combined = {**original, **supplemental}
    for sid, path in combined.items():
        target = overlay / ("1-tumor" if expected[sid] else "0-normal") / f"{sid}.csv"
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.is_symlink():
            if target.resolve() != path.resolve():
                raise ValueError(f"Stale overlay link: {target}")
        elif target.exists():
            raise ValueError(f"Overlay path already occupied: {target}")
        else:
            target.symlink_to(path.resolve())
    return overlay, missing


In [ ]:
from pathlib import Path
import json, subprocess, sys, zipfile

slugs = {
    "uniscale": "camelyon16-precomputedfeatures-uniscale",
    "multiscale": "camelyon16-precomputedfeatures-multiscale",
    "uni": "camelyon16-uni-uniscale-preprocessed-features",
}
assert SOURCE in slugs, SOURCE
input_mounts = [p for p in Path("/kaggle/input").iterdir() if p.is_dir()]
if SOURCE == "uni":
    feature_roots = [p.parent for p in Path("/kaggle/input").rglob("embeddings")
                     if p.is_dir() and (p.parent / "patches" / "patches").is_dir()]
else:
    feature_roots = [p.parent for p in Path("/kaggle/input").rglob("0-normal")
                     if p.is_dir() and (p.parent / "1-tumor").is_dir()]
    if SOURCE == "multiscale":
        feature_roots = [p for p in feature_roots if "cat" in p.parts]
    else:
        feature_roots = [p for p in feature_roots if "cat" not in p.parts and "fuse" not in p.parts]
assert len(feature_roots) == 1, f"Expected one {SOURCE} feature folder; found {feature_roots}; mounted folders: {[p.name for p in input_mounts]}"
input_root = feature_roots[0]
code_matches = [p for p in Path("/kaggle/input").rglob("prepare.py") if (p.parent / "reference_slides.csv").is_file()]
if len(code_matches) == 1:
    code_dir = code_matches[0].parent
else:
    bundles = list(Path("/kaggle/input").rglob("preprocessing_bundle.zip"))
    assert len(bundles) == 1, f"Cannot locate preprocessing code; found scripts={code_matches}, bundles={bundles}"
    code_dir = Path("/kaggle/working/camelyon16_preprocessing_code")
    code_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundles[0]) as bundle:
        bundle.extractall(code_dir)
if SOURCE == "multiscale":
    sys.path.insert(0, "/kaggle/working")
    from multiscale_overlay import overlay_multiscale
    input_root, supplemented = overlay_multiscale(
        input_root, Path("/kaggle/input"), code_dir / "reference_slides.csv",
        Path("/kaggle/temp/camelyon16_multiscale_combined"))
    print(f"Combined original dataset with {len(supplemented)} supplemental slides: {supplemented}")
output_root = Path("/kaggle/working") / f"camelyon16-{SOURCE}-prepared-v1"
command = [sys.executable, str(code_dir / "prepare.py"), "--source", SOURCE, "--input-root", str(input_root), "--output-root", str(output_root), "--seed", str(SEED)]
print("Preparing", SOURCE, "from", input_root, flush=True)
subprocess.run(command, check=True)


In [ ]:
import pandas as pd
summary = json.loads((output_root / "PREPROCESSING_COMPLETE.json").read_text())
folds = pd.read_csv(output_root / "folds.csv")
slides = pd.read_csv(output_root / "slides.csv")
assert len(slides) == 399 and len(list((output_root / "bags").glob("*.npy"))) == 399
assert folds["official_split"].value_counts().to_dict() == {"train": 270, "test": 129}
assert folds.loc[folds.official_split == "test", "validation_fold"].isna().all()
print(summary)
display(pd.crosstab(folds.validation_fold, folds.label))
print("Prepared output:", output_root)
print("Choose Save Version → Save & Run All, wait for completion, then check the saved Output tab. Quick Save is insufficient.")
